# Solution D2 · What PCA sees in the digits: pixels, centroids and lost information

**Type**: Interpret

## Objective

Read a PCA on images: which pixels never vary, where the variance sits (center versus edge), how many components are needed for a given share of variance, how the classes arrange themselves in the reduced space (which digits get confused) and how much classification information
is lost by reducing to 2, 10 or 30 dimensions (with a centroid classifier).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
images = X.reshape(-1, 8, 8)
edge = np.zeros((8, 8), dtype=bool)
edge[[0, 7], :] = True
edge[:, [0, 7]] = True
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
pixel_var = X.var(0)
constant_pixels = int((pixel_var == 0).sum())
edge_variance_share = float(pixel_var.reshape(8, 8)[edge].sum() / pixel_var.sum())
print(f"pixels with zero variance: {constant_pixels} | share of the variance in the 28 edge pixels: {edge_variance_share:.1%}")

In [ ]:
pca = PCA().fit(X)
cumulative = np.cumsum(pca.explained_variance_ratio_)
k90, k99 = int(np.searchsorted(cumulative, 0.9)) + 1, int(np.searchsorted(cumulative, 0.99)) + 1
pc1_image = pca.components_[0].reshape(8, 8)
absolute = np.abs(pc1_image)
pc1_center = float(absolute[2:6, 2:6].mean() - absolute[edge].mean())
print(f"components for 90% and 99% of the variance: {k90} and {k99} | mean weight center - edge in PC1: {pc1_center:.3f}")
print(np.round(pc1_image, 2))

In [ ]:
P2 = PCA(2).fit_transform(X)
centroids = np.array([P2[y == c].mean(0) for c in range(10)])
D = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=2)
np.fill_diagonal(D, np.inf)
a, b = np.unravel_index(D.argmin(), D.shape)
closest_pair = (int(min(a, b)), int(max(a, b)))
isolated_class = int(D.min(axis=1).argmax())
print("pair of digits with the closest centroids:", closest_pair, "| most isolated digit:", isolated_class)
print("distance to the closest centroid, per digit:", D.min(axis=1).round(2))

In [ ]:
centroid_acc = {k: cross_val_score(make_pipeline(PCA(k), NearestCentroid()), X, y, cv=cv).mean() for k in [2, 10, 30, 64]}
print({k: round(v, 3) for k, v in centroid_acc.items()})

## Check

In [ ]:
with verify():
    check(constant_pixels == (X.var(0) == 0).sum() and constant_pixels >= 3, "some pixels (at the corners) never vary", "(X.var(0) == 0).sum()")
    check(close(edge_variance_share, X.reshape(-1, 64).var(0)[edge.ravel()].sum() / X.var(0).sum()) and edge_variance_share < 0.25, "the outer ring (28 pixels out of 64) contains less than a quarter of the total variance", "sum of the variances of the 28 edge pixels / total")
    check(k90 == int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.9)) + 1 and 18 <= k90 <= 24 and 38 <= k99 <= 48, "about 21 components for 90% and about 41 for 99% (out of 64 pixels)", "np.searchsorted(np.cumsum(explained_variance_ratio_), threshold) + 1")
    check(pc1_image.shape == (8, 8) and pc1_center > 0.05, "the first component weighs the central pixels more than the edge ones", "components_[0].reshape(8, 8)")
    check(centroids.shape == (10, 2) and close(centroids[3], P2[y == 3].mean(0)) and closest_pair[0] < closest_pair[1] and isolated_class == 0, "centroids computed; the digit 0 is the most isolated", "mean of the scores per class and distance matrix")
    check(set(centroid_acc) == {2, 10, 30, 64} and centroid_acc[2] < 0.65 and centroid_acc[10] > 0.85 and centroid_acc[64] > 0.88, "with 2 components a centroid classifier has less than 0.65; with 10 over 0.85; with 64 over 0.88", "NearestCentroid inside the pipeline with PCA(k)")
    check(centroid_acc[10] > centroid_acc[2] + 0.25, "going from 2 to 10 components gains over 25 points", "the 2D map loses a lot of class information")
done()

## What you learned

- **The pixels are coordinates:** 3 pixels (at the corners) never vary and carry no information, and the outer ring of 28 pixels (44% of the pixels) contains only 19% of the variance. The variance sits in the center of the image, where the strokes of the digits pass: the first component weighs mostly the center
  compared to the edge.
- **How much variance, how many components:** 21 components for 90% and 41 for 99% out of 64 pixels. In images the redundancy is very high (neighboring pixels are correlated) and the effective dimension is much smaller than the nominal one.
- **A 2-dimensional map loses a lot of class information:** with a centroid classifier you go from 0.61 with 2 components (the two dimensions capture 29% of the variance) to 0.88 with 10. Some digits stay far from the others (the 0 is the most isolated) and others overlap:
  reading a 2D map with too much confidence leads to concluding that "the classes separate" or "don't separate" because of an artifact of the projection.
- **The number of components for the variance is not the one for classification**: with 30 components (96% of the variance) the centroid accuracy (0.90) is equal to that with all 64 pixels. For visualization 2 are enough, for classification 20-30.

## Common pitfall

Judging the separability of the classes from a plot of the first two components. Classes that overlap in 2D can be well separated in 10 dimensions (and vice versa, less often): the map is a projection, not a measurement.

## Interview question

<details><summary>A 2D PCA plot shows overlapping classes. What do you conclude?</summary>

Little: two components capture a fraction of the variance (here 29%) and the overlap can be an artifact of the projection. I measure the separability with a classifier in more dimensions (cross-validation) and use a nonlinear technique (t-SNE, UMAP) only
to explore, remembering that the distances in those maps are not quantitative.

</details>